# Auditoria de hiperparâmetros
Lê todos os `resolved_config.yaml` em `models/` e monta uma tabela comparativa.

In [37]:
import json
from pathlib import Path
import yaml, pandas as pd

ROOT = Path("/teamspace/lightning_storage/models")
paths = [ROOT/'lora-v3',
         ROOT/'lora-match-seed42',
         ROOT/'lora-match-seed43-v2',
         ROOT/'dora',
         ROOT/'dora-seed43',
         # ROOT/'qdora',
         # ROOT/'qdora-seed43',
         ROOT/'qdora-seed42-fp32',
         ROOT/'qdora-seed43-fp32',
         ROOT/'qlora',
         ROOT/'qlora-seed43']

In [38]:
with open('outputs/final_path_versions.txt', 'w') as wf:
    wf.write(json.dumps([[str(path) for path in paths]]))

### Helpers

In [39]:
def flatten(d, prefix=""):
    out = {}
    for k, v in (d or {}).items():
        key = f"{prefix}{k}"
        if isinstance(v, dict):
            out.update(flatten(v, key + "."))
        else:
            out[key] = str(v) if isinstance(v, (list, tuple)) else v
    return out

# runs = {str(p).split('/')[-1]: flatten(yaml.safe_load((p/'resolved_config.yaml').read_text())) for p in paths}

def load_json_file(filename):
    with open(filename, 'r') as rf:
        res = json.load(rf)
    return res


### Train

In [40]:
full = {}
parameters = {}
devices = {}
adapter = {}
quant = {}
for p in paths:
    run_name = str(p).split('/')[-1]
    complete_file = load_json_file(p/'run_metadata.json')
    full[run_name] = complete_file
    parameters[run_name] = complete_file['training']
    devices[run_name]= complete_file['cuda_devices'][0]
    adapter[run_name] = complete_file['adapter']
    quant[run_name] = complete_file['quantization']


df = pd.DataFrame(full).sort_index()
df_params = pd.DataFrame(parameters).sort_index()
df_params = df_params.loc[df_params.index != 'resume_from_checkpoint', :]
df_devices = pd.DataFrame(devices).sort_index()
df_adapter = pd.DataFrame(adapter).sort_index()
df_quant = pd.DataFrame(quant)

df

,lora-v3,lora-match-seed42,lora-match-seed43-v2,dora,dora-seed43,qdora-seed42-fp32,qdora-seed43-fp32,qlora,qlora-seed43
adapter,"{'rank': 32, 'alpha': 64, 'target_modules': ['...","{'rank': 32, 'alpha': 64, 'target_modules': ['...","{'rank': 32, 'alpha': 64, 'target_modules': ['...","{'rank': 32, 'alpha': 64, 'target_modules': ['...","{'rank': 32, 'alpha': 64, 'target_modules': ['...","{'rank': 32, 'alpha': 64, 'target_modules': ['...","{'rank': 32, 'alpha': 64, 'target_modules': ['...","{'rank': 32, 'alpha': 64, 'target_modules': ['...","{'rank': 32, 'alpha': 64, 'target_modules': ['..."
cuda_devices,"[{'index': 0, 'name': 'NVIDIA H100 80GB HBM3',...","[{'index': 0, 'name': 'NVIDIA H100 80GB HBM3',...","[{'index': 0, 'name': 'NVIDIA H100 80GB HBM3',...","[{'index': 0, 'name': 'NVIDIA H100 80GB HBM3',...","[{'index': 0, 'name': 'NVIDIA H100 80GB HBM3',...","[{'index': 0, 'name': 'NVIDIA H100 80GB HBM3',...","[{'index': 0, 'name': 'NVIDIA H100 80GB HBM3',...","[{'index': 0, 'name': 'NVIDIA H100 80GB HBM3',...","[{'index': 0, 'name': 'NVIDIA H100 80GB HBM3',..."
experiment,"{'project': 'cs7643-dora-lora-quantization', '...","{'project': 'cs7643-dora-lora-quantization', '...","{'project': 'cs7643-dora-lora-quantization', '...","{'project': 'cs7643-dora-lora-quantization', '...","{'project': 'cs7643-dora-lora-quantization', '...","{'project': 'cs7643-dora-lora-quantization', '...","{'project': 'cs7643-dora-lora-quantization', '...","{'project': 'cs7643-dora-lora-quantization', '...","{'project': 'cs7643-dora-lora-quantization', '..."
model,"{'id': 'meta-llama/Meta-Llama-3-8B', 'torch_dt...","{'id': 'meta-llama/Meta-Llama-3-8B', 'torch_dt...","{'id': 'meta-llama/Meta-Llama-3-8B', 'torch_dt...","{'id': 'meta-llama/Meta-Llama-3-8B', 'torch_dt...","{'id': 'meta-llama/Meta-Llama-3-8B', 'torch_dt...","{'id': 'meta-llama/Meta-Llama-3-8B', 'torch_dt...","{'id': 'meta-llama/Meta-Llama-3-8B', 'torch_dt...","{'id': 'meta-llama/Meta-Llama-3-8B', 'torch_dt...","{'id': 'meta-llama/Meta-Llama-3-8B', 'torch_dt..."
num_eval_examples,0,0,0,0,0,0,0,0,0
num_train_examples,170420,170420,170420,170420,170420,170420,170420,170420,170420
quantization,"{'enabled': False, 'bits': 4, 'quant_type': 'n...","{'enabled': False, 'bits': 4, 'quant_type': 'n...","{'enabled': False, 'bits': 4, 'quant_type': 'n...","{'enabled': False, 'bits': 4, 'quant_type': 'n...","{'enabled': False, 'bits': 4, 'quant_type': 'n...","{'enabled': True, 'bits': 4, 'quant_type': 'nf...","{'enabled': True, 'bits': 4, 'quant_type': 'nf...","{'enabled': True, 'bits': 4, 'quant_type': 'nf...","{'enabled': True, 'bits': 4, 'quant_type': 'nf..."
total_params,8086884352,8086884352,8086884352,8087670784,8087670784,4598009856,4598009856,4597223424,4597223424
trainable_param_percent,0.700184,0.700184,0.700184,0.70984,0.70984,1.248574,1.248574,1.23168,1.23168
trainable_params,56623104,56623104,56623104,57409536,57409536,57409536,57409536,56623104,56623104


In [41]:
df_params

,lora-v3,lora-match-seed42,lora-match-seed43-v2,dora,dora-seed43,qdora-seed42-fp32,qdora-seed43-fp32,qlora,qlora-seed43
bf16,True,True,True,True,True,True,True,True,True
dataloader_num_workers,0,0,0,0,0,0,0,0,0
dataset_text_field,text,text,text,text,text,text,text,text,text
fp16,False,False,False,False,False,False,False,False,False
gradient_accumulation_steps,1,1,1,1,1,1,1,1,1
gradient_checkpointing,True,True,True,True,True,True,True,True,True
gradient_checkpointing_kwargs,{'use_reentrant': False},{'use_reentrant': False},{'use_reentrant': False},{'use_reentrant': False},{'use_reentrant': False},{'use_reentrant': False},{'use_reentrant': False},{'use_reentrant': False},{'use_reentrant': False}
learning_rate,0.0003,0.0001,0.0001,0.0001,0.0001,0.0001,0.0001,0.0001,0.0001
logging_dir,None,None,None,None,None,None,None,None,None
logging_steps,50,50,50,50,50,50,50,50,50


In [42]:
df_params

,lora-v3,lora-match-seed42,lora-match-seed43-v2,dora,dora-seed43,qdora-seed42-fp32,qdora-seed43-fp32,qlora,qlora-seed43
bf16,True,True,True,True,True,True,True,True,True
dataloader_num_workers,0,0,0,0,0,0,0,0,0
dataset_text_field,text,text,text,text,text,text,text,text,text
fp16,False,False,False,False,False,False,False,False,False
gradient_accumulation_steps,1,1,1,1,1,1,1,1,1
gradient_checkpointing,True,True,True,True,True,True,True,True,True
gradient_checkpointing_kwargs,{'use_reentrant': False},{'use_reentrant': False},{'use_reentrant': False},{'use_reentrant': False},{'use_reentrant': False},{'use_reentrant': False},{'use_reentrant': False},{'use_reentrant': False},{'use_reentrant': False}
learning_rate,0.0003,0.0001,0.0001,0.0001,0.0001,0.0001,0.0001,0.0001,0.0001
logging_dir,None,None,None,None,None,None,None,None,None
logging_steps,50,50,50,50,50,50,50,50,50


In [43]:
df_devices

,lora-v3,lora-match-seed42,lora-match-seed43-v2,dora,dora-seed43,qdora-seed42-fp32,qdora-seed43-fp32,qlora,qlora-seed43
index,0,0,0,0,0,0,0,0,0
major,9,9,9,9,9,9,9,9,9
minor,0,0,0,0,0,0,0,0,0
name,NVIDIA H100 80GB HBM3,NVIDIA H100 80GB HBM3,NVIDIA H100 80GB HBM3,NVIDIA H100 80GB HBM3,NVIDIA H100 80GB HBM3,NVIDIA H100 80GB HBM3,NVIDIA H100 80GB HBM3,NVIDIA H100 80GB HBM3,NVIDIA H100 80GB HBM3
total_memory_gb,79.178711,79.178711,79.178711,79.178711,79.178711,79.178711,79.178711,79.178711,79.178711


In [44]:
df_adapter

,lora-v3,lora-match-seed42,lora-match-seed43-v2,dora,dora-seed43,qdora-seed42-fp32,qdora-seed43-fp32,qlora,qlora-seed43
alpha,64,64,64,64,64,64,64,64,64
bias,none,none,none,none,none,none,none,none,none
dropout,0.05,0.05,0.05,0.05,0.05,0.05,0.05,0.05,0.05
magnitude_dtype,NaN,NaN,NaN,NaN,NaN,fp32,fp32,NaN,NaN
rank,32,32,32,32,32,32,32,32,32
target_modules,"[q_proj, k_proj, v_proj, up_proj, down_proj]","[q_proj, k_proj, v_proj, up_proj, down_proj]","[q_proj, k_proj, v_proj, up_proj, down_proj]","[q_proj, k_proj, v_proj, up_proj, down_proj]","[q_proj, k_proj, v_proj, up_proj, down_proj]","[q_proj, k_proj, v_proj, up_proj, down_proj]","[q_proj, k_proj, v_proj, up_proj, down_proj]","[q_proj, k_proj, v_proj, up_proj, down_proj]","[q_proj, k_proj, v_proj, up_proj, down_proj]"
task_type,CAUSAL_LM,CAUSAL_LM,CAUSAL_LM,CAUSAL_LM,CAUSAL_LM,CAUSAL_LM,CAUSAL_LM,CAUSAL_LM,CAUSAL_LM
use_dora,False,False,False,True,True,True,True,False,False


In [45]:
df_quant

,lora-v3,lora-match-seed42,lora-match-seed43-v2,dora,dora-seed43,qdora-seed42-fp32,qdora-seed43-fp32,qlora,qlora-seed43
enabled,False,False,False,False,False,True,True,True,True
bits,4,4,4,4,4,4,4,4,4
quant_type,nf4,nf4,nf4,nf4,nf4,nf4,nf4,nf4,nf4
double_quant,True,True,True,True,True,True,True,True,True


In [46]:
df_adapter.to_csv('outputs/audit_adapters.csv')
df_devices.to_csv('outputs/audit_devices.csv')
df_params.to_csv('outputs/audit_params.csv')
df_quant.to_csv('outputs/audit_quantization.csv')

### Evals

In [47]:
import os

In [48]:
def pick_resolved_eval_config(path):
    dir_content = os.listdir(path/'paper_eval_results')
    if 'eval_resolved_config.yaml' in dir_content:
        file_path = path/'paper_eval_results'/'eval_resolved_config.yaml'
    
    else:
        file_path = path/'paper_eval_results'/dir_content[-1]/'eval_resolved_config.yaml' # naming standard should sort by recency
        # subdir_content = os.listdir(dir_content[-1]) 
        print('picked run: {}, among all {}'.format(dir_content[-1], ', '.join(dir_content)))
    
    return file_path

In [49]:
excluded = []

full = {}
evals = {}
for p in paths:
    run_name = str(p).split('/')[-1]

    if run_name not in excluded:
        print(p)
        file_path = pick_resolved_eval_config(p)
        
        complete_file = yaml.safe_load(file_path.read_text())
        full[run_name] = complete_file
        evals[run_name] = complete_file['eval']
        # devices[run_name]= complete_file['cuda_devices']
        # adapter[run_name] = complete_file['adapter']


df = pd.DataFrame(full).sort_index()
df_evals = pd.DataFrame(evals).sort_index()
df_evals = df_evals.iloc[1:,:] # drop adapter_path 

# df_devices = pd.DataFrame(devices).sort_index()
# df_adapter = pd.DataFrame(adapter).sort_index()

# df

/teamspace/lightning_storage/models/lora-v3
picked run: paper_eval_20260827_103908, among all paper_eval_20260827_103908
/teamspace/lightning_storage/models/lora-match-seed42
picked run: paper_eval_20260717_002852, among all paper_eval_20260712_201111, paper_eval_20260717_002852
/teamspace/lightning_storage/models/lora-match-seed43-v2
picked run: paper_eval_20260820_194824, among all paper_eval_20260820_194824
/teamspace/lightning_storage/models/dora
picked run: paper_eval_20260827_022522, among all paper_eval_20260710_192428, paper_eval_20260827_022522
/teamspace/lightning_storage/models/dora-seed43
picked run: paper_eval_20260722_130811, among all paper_eval_20260722_130811
/teamspace/lightning_storage/models/qdora-seed42-fp32
picked run: paper_eval_20260825_015812, among all paper_eval_20260825_015812
/teamspace/lightning_storage/models/qdora-seed43-fp32
picked run: paper_eval_20260825_015842, among all paper_eval_20260825_015842
/teamspace/lightning_storage/models/qlora
picked run:

In [50]:
df_evals

,lora-v3,lora-match-seed42,lora-match-seed43-v2,dora,dora-seed43,qdora-seed42-fp32,qdora-seed43-fp32,qlora,qlora-seed43
allow_base_model_eval,False,False,False,False,False,False,False,False,False
batch_size,1,1,1,1,1,1,1,1,1
do_sample,False,False,False,False,False,False,False,False,False
limit,None,None,None,None,None,None,None,None,None
max_new_tokens,32,32,32,32,32,32,32,32,32
num_beams,4,4,4,4,4,4,4,4,4
save_every_n_batches,1,1,1,1,1,1,1,1,1
task_url_template,https://raw.githubusercontent.com/AGI-Edgerunn...,https://raw.githubusercontent.com/AGI-Edgerunn...,https://raw.githubusercontent.com/AGI-Edgerunn...,https://raw.githubusercontent.com/AGI-Edgerunn...,https://raw.githubusercontent.com/AGI-Edgerunn...,https://raw.githubusercontent.com/AGI-Edgerunn...,https://raw.githubusercontent.com/AGI-Edgerunn...,https://raw.githubusercontent.com/AGI-Edgerunn...,https://raw.githubusercontent.com/AGI-Edgerunn...
tasks,"[boolq, piqa, social_i_qa, hellaswag, winogran...","[boolq, piqa, social_i_qa, hellaswag, winogran...","[boolq, piqa, social_i_qa, hellaswag, winogran...","[boolq, piqa, social_i_qa, hellaswag, winogran...","[boolq, piqa, social_i_qa, hellaswag, winogran...","[boolq, piqa, social_i_qa, hellaswag, winogran...","[boolq, piqa, social_i_qa, hellaswag, winogran...","[boolq, piqa, social_i_qa, hellaswag, winogran...","[boolq, piqa, social_i_qa, hellaswag, winogran..."
temperature,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1


In [51]:
df_evals.to_csv('outputs/audit_evals.csv')